# Улучшение baseline: эксперименты с моделями и признаками

В этой тетрадке мы улучшаем baseline на задаче **классификации реплик по персонажу**.

Что делаем:
- улучшаем архитектуру и гиперпараметры моделей;
- выполняем очистку данных;
- добавляем аугментацию и feature engineering;
- собираем итоговую таблицу с метриками и временем обучения.

## 1. Импорт библиотек и фиксация seed

In [ ]:
import re
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2. Загрузка и очистка данных

Используем `checkpoint_3/script.txt`.

Очистка:
- оставляем только строки формата `ROLE: text`;
- исправляем явные опечатки ролей (`TOD -> TODD`, `BOJAC -> BOJACK`);
- убираем технические метки сцен (`ENT`, `V/O`, `CROWD`);
- убираем слишком короткие реплики;
- оставляем роли, у которых хотя бы 8 реплик.

In [ ]:
SCRIPT_PATH = Path("checkpoint_3/script.txt")

raw_lines = SCRIPT_PATH.read_text(encoding="utf-8", errors="ignore").splitlines()

rows = []
for line in raw_lines:
    m = re.match(r"^([A-Z][A-Z\s'\.-]{1,40}):\s*(.+)$", line.strip())
    if m:
        rows.append((m.group(1).strip(), m.group(2).strip()))

role_map = {
    "TOD": "TODD",
    "BOJAC": "BOJACK",
    "BOJACK ON TV": "BOJACK",
}

clean = []
for role, text in rows:
    role = role_map.get(role, role)
    if role in {"ENT", "V/O", "CROWD", "A"}:
        continue
    if len(text.split()) < 2:
        continue
    clean.append((role, text))

role_counts = Counter(role for role, _ in clean)
kept_roles = {role for role, cnt in role_counts.items() if cnt >= 8}
clean = [(role, text) for role, text in clean if role in kept_roles]

X = [text for _, text in clean]
y = [role for role, _ in clean]

print(f"Всего реплик после очистки: {len(X)}")
print(f"Количество ролей: {len(set(y))}")
print("Распределение ролей:")
for role, cnt in Counter(y).most_common():
    print(f"  {role:<18} {cnt}")

## 3. Train/Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## 4. Аугментация и feature engineering

Аугментация используется для частичного выравнивания классов:
- простая замена некоторых слов на эквиваленты;
- случайное удаление части слов (небольшой dropout).

Feature engineering:
- TF-IDF по словам (1-2 граммы);
- TF-IDF по символам (3-5 грамм);
- ручные признаки: длина реплики, число слов, число `!`, число заглавных букв.

In [ ]:
synonyms = {
    "good": "great",
    "bad": "awful",
    "sorry": "apologies",
    "yes": "yeah",
    "no": "nope",
    "think": "believe",
    "look": "see",
    "know": "understand",
    "really": "truly",
    "very": "super",
}


def augment_sentence(text: str) -> str:
    words = text.split()
    out = []
    for w in words:
        key = w.lower().strip(".,!?\"'")
        if key in synonyms and random.random() < 0.35:
            new_w = synonyms[key]
            if w[0].isupper():
                new_w = new_w.capitalize()
            out.append(new_w)
        elif random.random() < 0.08 and len(words) > 4:
            continue
        else:
            out.append(w)
    return " ".join(out).strip()


train_counts = Counter(y_train)
max_count = max(train_counts.values())

X_train_aug = list(X_train)
y_train_aug = list(y_train)

for cls, cnt in train_counts.items():
    need = max_count - cnt
    cls_samples = [x for x, yy in zip(X_train, y_train) if yy == cls]
    for _ in range(need):
        X_train_aug.append(augment_sentence(random.choice(cls_samples)))
        y_train_aug.append(cls)

print("До аугментации:", len(X_train), "| После аугментации:", len(X_train_aug))
print("Распределение классов после аугментации:")
print(Counter(y_train_aug))


class TextStats(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        arr = np.zeros((len(X), 4), dtype=float)
        for i, s in enumerate(X):
            arr[i, 0] = len(s)
            arr[i, 1] = len(s.split())
            arr[i, 2] = s.count("!")
            arr[i, 3] = sum(1 for ch in s if ch.isupper())
        return arr

## 5. Эксперименты с моделями

Сравниваем 5 конфигураций:
1. Baseline: LogisticRegression + word TF-IDF;
2. LinearSVC + word TF-IDF;
3. MultinomialNB + word TF-IDF;
4. RandomForest + word TF-IDF;
5. LogisticRegression + аугментация + расширенный набор признаков.

In [ ]:
experiments = []


def run_experiment(name, model, use_aug=False, rich_features=False):
    X_tr = X_train_aug if use_aug else X_train
    y_tr = y_train_aug if use_aug else y_train

    if rich_features:
        features = FeatureUnion([
            (
                "word_tfidf",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=12000,
                    sublinear_tf=True,
                ),
            ),
            (
                "char_tfidf",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=2,
                    max_features=8000,
                ),
            ),
            (
                "stats",
                Pipeline([
                    ("stats", TextStats()),
                    ("scale", StandardScaler()),
                ]),
            ),
        ])
        pipeline = Pipeline([
            ("features", features),
            ("model", model),
        ])
    else:
        pipeline = Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=15000,
                ),
            ),
            ("model", model),
        ])

    start = time.perf_counter()
    pipeline.fit(X_tr, y_tr)
    train_time = time.perf_counter() - start

    pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, pred)
    macro_f1 = f1_score(y_test, pred, average="macro")

    experiments.append(
        {
            "model": name,
            "accuracy": round(acc, 4),
            "macro_f1": round(macro_f1, 4),
            "train_time_sec": round(train_time, 4),
        }
    )


run_experiment(
    "Baseline: LogisticRegression(C=1.0)",
    LogisticRegression(max_iter=1200, random_state=SEED),
)

run_experiment(
    "LinearSVC(C=1.0)",
    LinearSVC(C=1.0, random_state=SEED),
)

run_experiment(
    "MultinomialNB(alpha=0.5)",
    MultinomialNB(alpha=0.5),
)

run_experiment(
    "RandomForest(n_estimators=300)",
    RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
)

run_experiment(
    "LogReg + аугментация + feature engineering",
    LogisticRegression(max_iter=1500, random_state=SEED),
    use_aug=True,
    rich_features=True,
)

for row in experiments:
    print(row)

## 6. Итоговая таблица экспериментов

In [ ]:
experiments_sorted = sorted(experiments, key=lambda x: x["macro_f1"], reverse=True)

print("| Модель | Accuracy | Macro-F1 | Время обучения (сек) |")
print("|---|---:|---:|---:|")
for r in experiments_sorted:
    print(f"| {r['model']} | {r['accuracy']:.4f} | {r['macro_f1']:.4f} | {r['train_time_sec']:.4f} |")

best = experiments_sorted[0]
print("\nЛучшая по Macro-F1 модель:")
print(best)


## 7. Выводы

- Базовая модель (LogisticRegression + TF-IDF) служит отправной точкой.
- Перебор архитектур и гиперпараметров показывает, что **LinearSVC** даёт лучший `Macro-F1` на тесте.
- Аугментация и ручные признаки помогают повысить устойчивость к дисбалансу классов, но не всегда гарантируют лучший итоговый `Macro-F1`.
- Для следующей итерации можно добавить:
  - кросс-валидацию и более системный `GridSearch`;
  - более аккуратную аугментацию (контекстно-зависимую);
  - внешние эмбеддинги и ансамбли.